In [1]:
import numpy as np
import pandas as pd
from thyrosim_model import Thyrosim, run_simulation, compute_ft4_ft3

In [2]:
# Create parameters
def create_default_params():
    params = {}
    for i in range(1,49):
        params[f"p{i}"] = 1.0
    return params

In [3]:
# Simulate
def simulate_rtf(rtf):

    params = create_default_params()

    # Assume RTF corresponds to parameter p1
    params["p1"] = rtf

    dial = [1,1,1,1]
    inf = [0,0]

    model = Thyrosim(
        dial=dial,
        inf=inf,
        kdelay=0.1,
        params=params
    )

    initial_conditions = np.ones(19)

    t, states = run_simulation(model, 0, 200, initial_conditions)

    ft4, ft3 = compute_ft4_ft3(states, params)

    tsh = states[6]

    # return steady state averages
    return np.mean(ft4[-20:]), np.mean(ft3[-20:]), np.mean(tsh[-20:])

In [4]:
# Create "fake" dataset
def generate_dataset():

    rtf_values = np.linspace(0.1,1.0,20)

    data = []

    for rtf in rtf_values:

        ft4, ft3, tsh = simulate_rtf(rtf)

        data.append([rtf, ft4, ft3, tsh])

    df = pd.DataFrame(data, columns=["RTF","FT4","FT3","TSH"])

    return df

In [5]:
# grid search
def estimate_rtf(observed_ft4, observed_ft3, observed_tsh, dataset):

    errors = []

    for _, row in dataset.iterrows():

        err = (
            (row.FT4 - observed_ft4)**2 +
            (row.FT3 - observed_ft3)**2 +
            (row.TSH - observed_tsh)**2
        )

        errors.append(err)

    best_index = np.argmin(errors)

    return dataset.iloc[best_index]["RTF"]

In [6]:
dataset = generate_dataset()

print(dataset.head())

# pretend these are observed patient values
true_rtf = 0.6

ft4, ft3, tsh = simulate_rtf(true_rtf)

print("\nSynthetic patient values:")
print("FT4:", ft4)
print("FT3:", ft3)
print("TSH:", tsh)

estimated_rtf = estimate_rtf(ft4, ft3, tsh, dataset)

print("\nTrue RTF:", true_rtf)
print("Estimated RTF:", estimated_rtf)

        RTF       FT4       FT3       TSH
0  0.100000  0.080498  2.268282  0.103994
1  0.147368  0.114715  2.247841  0.108815
2  0.194737  0.147196  2.235288  0.093527
3  0.242105  0.178365  2.229122  0.095816
4  0.289474  0.208475  2.228204  0.090799

Synthetic patient values:
FT4: 0.3922701317145444
FT3: 2.3080819162579833
TSH: 0.06611036020702721

True RTF: 0.6
Estimated RTF: 0.6210526315789474
